In [ ]:
import pandas as pd
import numpy as np
import missingno as msno

df = pd.read_csv("balanced_patient_data.csv")

print(df.shape)

df = df.sort_values(["patient_id", "ICULOS"])
missing_before = df.isnull().sum().sum()
print("Total missing values:", missing_before)

labs=['BaseExcess','HCO3','FiO2','pH','PaCO2','SaO2','AST','BUN','Alkalinephos','Calcium','Chloride','Creatinine','Bilirubin_direct','Glucose','Lactate',
      'Magnesium','Phosphate','Potassium','Bilirubin_total','TroponinI','Hct','Hgb','PTT','WBC','Fibrinogen','Platelets']

vitals = ['HR','O2Sat','Temp','SBP','MAP','DBP','Resp','EtCO2']

demographics = ['Age','Gender','Unit1','Unit2','HospAdmTime','ICULOS']

labels = ['SepsisLabel']

In [ ]:
# Replace string 'NaN' with actual np.nan if needed
df.replace("NaN", np.nan, inplace=True)

# Convert numeric columns properly
for col in df.columns:
    if col != "patient_id":
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Ensure SepsisLabel is binary int
df["SepsisLabel"] = df["SepsisLabel"].fillna(0).astype(int)

# Explore before Imputation

In [ ]:
labs_df = df[labs]
vitals_df = df[vitals]
demogs_df = df[demographics]

print("Percentage of NULL values in Labs:", np.mean((labs_df.isnull().sum() / labs_df.shape[0])))
print("Percentage of NULL values in Vitals:", np.mean((vitals_df.isnull().sum() / vitals_df.shape[0])))
print("Percentage of NULL values in Demographics:", np.mean((demogs_df.isnull().sum() / demogs_df.shape[0])))

msno.matrix(labs_df)

In [ ]:
msno.matrix(vitals_df)

In [ ]:
msno.matrix(demogs_df)

In [ ]:
missingdata_df = df.columns[df.isnull().any()].tolist()
msno.bar(df[missingdata_df], color="blue", log=False, figsize=(30,18))

In [ ]:
msno.heatmap(df[missingdata_df], figsize=(20,20)) # identifying correlations between missing values across columns.

In [ ]:
df[vitals + labs] = (
    df.groupby("patient_id")[vitals + labs]
    .ffill()
)

for col in vitals + labs:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

In [ ]:
df[demographics] = (
    df.groupby("patient_id")[demographics]
    .ffill()
    .bfill()
)

In [ ]:
missing_after = df.isnull().sum().sum()
print("Missing after:", missing_after)

In [ ]:
labs_df = df[labs]
vitals_df = df[vitals]
demogs_df = df[demographics]

print("Percentage of NULL values in Labs:", np.mean((labs_df.isnull().sum() / labs_df.shape[0])))
print("Percentage of NULL values in Vitals:", np.mean((vitals_df.isnull().sum() / vitals_df.shape[0])))
print("Percentage of NULL values in Demographics:", np.mean((demogs_df.isnull().sum() / demogs_df.shape[0])))

In [ ]:
missingdata_df = df.columns[df.isnull().any()].tolist()
msno.bar(df[missingdata_df], color="blue", log=False, figsize=(30,18))

In [ ]:
# df.to_csv("imputed_dataset.csv",index=False)